导包

In [1]:
from skimage import io, measure
import numpy as np
from preclassify import dicomp, hcluster
import random
from collections import  Counter
import torch.nn as nn
import torch
import torch.optim as optim
import torch.nn.functional as F
import cv2
import matplotlib.pyplot as plt
import os
import sys
import time
import math
from torch.autograd import Variable

超参设置

In [2]:
# 获取图片路径
im1_path  = 'sar_pic/Sulzberger2_1.bmp'
im2_path  = 'sar_pic/Sulzberger2_2.bmp'
imgt_path = 'sar_pic/Sulzberger2_gt.bmp'

# im1_path  = 'sar_pic/Chaohu1_1.bmp'
# im2_path  = 'sar_pic/Chaohu1_2.bmp'
# imgt_path = 'sar_pic/Chaohu1_gt.bmp'

# im1_path  = 'sar_pic/Chaohu2_1.bmp'
# im2_path  = 'sar_pic/Chaohu2_2.bmp'
# imgt_path = 'sar_pic/Chaohu2_gt.bmp'

# 用于训练的每个patch大小
patch_size = 9
# 设置训练集样本数
train_len = 10000
# 未改变和改变的训练集样本数
# 按比例选取
label_1_train_len = 8800

定义各类方法

In [3]:
# padding操作
def image_padding(data,r):
    if len(data.shape)==3:
        data_new=np.pad(data,((r,r),(r,r),(0,0)),'constant',constant_values=0)
        return data_new
    if len(data.shape)==2:
        data_new=np.pad(data,r,'constant',constant_values=0)
        return data_new

# 生成自然数数组并打乱
def arr(length):
  arr=np.arange(length-1)
  random.shuffle(arr)
  return arr

# 制作训练集
# 在每个像素周围提取 patch ，然后创建成符合 pytorch 处理的格式
def createTrainingCubes(X, y, patch_size):
    # 给 X 做 padding
    margin = int((patch_size - 1) / 2)
    zeroPaddedX = image_padding(X, margin)
    # 把类别 uncertainty 的像素忽略
    ele_num1 = np.sum(y==1)
    ele_num2 = np.sum(y==2)
    # num1和num2的比例决定1、2随机选取样本的比例
    patchesData_1 = np.zeros( (ele_num1, patch_size, patch_size, X.shape[2]) )
    patchesLabels_1 = np.zeros(ele_num1)
    print(ele_num1)
    print(ele_num2)
    patchesData_2 = np.zeros((ele_num2, patch_size, patch_size, X.shape[2]))
    patchesLabels_2 = np.zeros(ele_num2)

    patchIndex_1 = 0
    patchIndex_2 = 0
    for r in range(margin, zeroPaddedX.shape[0] - margin):
        for c in range(margin, zeroPaddedX.shape[1] - margin):
            # 把类别 uncertainty 的像素忽略
            if y[r-margin, c-margin] == 1 :
                patch_1 = zeroPaddedX[r - margin:r + margin + 1, c - margin:c + margin + 1]   
                patchesData_1[patchIndex_1, :, :, :] = patch_1
                patchesLabels_1[patchIndex_1] = y[r-margin, c-margin]
                patchIndex_1 = patchIndex_1 + 1
            elif y[r-margin, c-margin] == 2 :
                patch_2 = zeroPaddedX[r - margin:r + margin + 1, c - margin:c + margin + 1]   
                patchesData_2[patchIndex_2, :, :, :] = patch_2
                patchesLabels_2[patchIndex_2] = y[r-margin, c-margin]
                patchIndex_2 = patchIndex_2 + 1
    patchesLabels_1 = patchesLabels_1-1
    patchesLabels_2 = patchesLabels_2-1

    arr_1=arr(len(patchesData_1))
    arr_2=arr(len(patchesData_2))
  
    pdata=np.zeros((train_len, patch_size, patch_size, X.shape[2]))
    plabels = np.zeros(train_len)
    #标签为1和2的按比例选取
    for i in range(label_1_train_len):
      pdata[i,:,:,:]=patchesData_1[arr_1[i],:,:,:]
      plabels[i]=patchesLabels_1[arr_1[i]]
    for j in range(label_1_train_len,train_len):
      pdata[j,:,:,:]=patchesData_2[arr_2[j-train_len//2],:,:,:]
      plabels[j]=patchesLabels_2[arr_2[j-train_len//2]]
    print(Counter(plabels))
    return pdata, plabels

# 制作测试集
def createTestingCubes(X, patch_size):
    # 给 X 做 padding
    margin = int((patch_size - 1) / 2)
    zeroPaddedX = image_padding(X, margin)
    patchesData = np.zeros( (X.shape[0]*X.shape[1], patch_size, patch_size, X.shape[2]) )
    patchIndex = 0
    for r in range(margin, zeroPaddedX.shape[0] - margin):
        for c in range(margin, zeroPaddedX.shape[1] - margin):
            patch = zeroPaddedX[r - margin:r + margin + 1, c - margin:c + margin + 1]   
            patchesData[patchIndex, :, :, :] = patch
            patchIndex = patchIndex + 1
    return patchesData

def postprocess(res):
    res_new = res
    res = measure.label(res, connectivity=2)
    num = res.max()
    for i in range(1, num+1):
        idy, idx = np.where(res==i)
        if len(idy) <= 20:
            res_new[idy, idx] = 0
    return res_new

def evaluate(gtImg, tstImg):
    gtImg[np.where(gtImg>128)] = 255
    gtImg[np.where(gtImg<128)] = 0
    tstImg[np.where(tstImg>128)] = 255
    tstImg[np.where(tstImg<128)] = 0
    [ylen, xlen] = gtImg.shape
    FA = 0
    MA = 0

    for j in range(0,ylen):
        for i in range(0,xlen):
            if gtImg[j,i]==0 and tstImg[j,i]!=0 :
                FA = FA+1
            if gtImg[j,i]!=0 and tstImg[j,i]==0 :
                MA = MA+1
  
    OE = FA+MA
    PCC = 1-OE/(ylen*xlen)
    
    # 计算 Kappa 系数
    PRE = ((FA + (ylen * xlen - OE - FA)) * (MA + (ylen * xlen - OE - FA)) + (MA + OE - MA) * (FA + OE - FA)) / (ylen * xlen) ** 2
    # 更准确的 Kappa 计算方式：
    # TP: 真阳性 (Changed AND Predicted Changed)
    # TN: 真阴性 (Unchanged AND Predicted Unchanged)
    # FP: 假阳性 (Unchanged BUT Predicted Changed) -> FA
    # FN: 假阴性 (Changed BUT Predicted Unchanged) -> MA
    TP = np.sum((gtImg != 0) & (tstImg != 0))
    TN = np.sum((gtImg == 0) & (tstImg == 0))
    FP = FA
    FN = MA
    
    Po = (TP + TN) / (ylen * xlen)
    Pe = ((TP + FP) * (TP + FN) + (FN + TN) * (FP + TN)) / (ylen * xlen) ** 2
    Kappa = (Po - Pe) / (1 - Pe)
    
    print(' Change detection results ==>')
    print(' ... ... FA:  ', FA)
    print(' ... ... MA:  ', MA)
    print(' ... ... OE:  ', OE)
    print(' ... ... PCC: ', format(PCC*100, '.2f'), '%')
    print(' ... ... Kappa: ', format(Kappa, '.4f'))


读取图片（包括原图1，原图2和人工标注差异图gt）

In [4]:
im1 = io.imread(im1_path)[:,:,0].astype(np.float32)
im2 = io.imread(im2_path)[:,:,0].astype(np.float32)
im_gt = io.imread(imgt_path)[:,:,0].astype(np.float32)
print(im1.shape)

(256, 256)


计算差分图

In [5]:
im_di = dicomp(im1, im2)
ylen, xlen = im_di.shape
pix_vec = im_di.reshape([ylen*xlen, 1])

分析差分图之 使用 “hiearchical FCM 聚类方法” 进行预分类

In [6]:
# 将高概率 未改变 的像素的标签设为 1
# 将高概率 改变 的像素的标签设为 2
# 将 不确定 的像素的标签设为1.5
preclassify_lab = hcluster(pix_vec, im_di)
print('... ... hiearchical clustering finished !!!')

... ... 1st round clustering ... ...
... ... 2nd round clustering ... ...
... ... hiearchical clustering finished !!!


制作训练集和测试集

In [7]:
# 将原图1、原图2和差分图放到一个3维数组里
mdata = np.zeros([im1.shape[0], im1.shape[1], 3], dtype=np.float32)
mdata[:,:,0] = im1
mdata[:,:,1] = im2
mdata[:,:,2] = im_di
mlabel = preclassify_lab

# 制作训练集
x_train, y_train = createTrainingCubes(mdata, mlabel, patch_size)
x_train = x_train.transpose(0, 3, 1, 2)
print('... x train shape: ', x_train.shape)
print('... y train shape: ', y_train.shape)

# 制作测试集
x_test = createTestingCubes(mdata, patch_size)
x_test = x_test.transpose(0, 3, 1, 2)
print('... x test shape: ', x_test.shape)

43669
13809
Counter({np.float64(0.0): 8800, np.float64(1.0): 1200})
... x train shape:  (10000, 3, 9, 9)
... y train shape:  (10000,)
... x test shape:  (65536, 3, 9, 9)


定义读取数据集的类

In [8]:
class TrainDS(torch.utils.data.Dataset): 
    def __init__(self):
        self.len = x_train.shape[0]
        self.x_data = torch.FloatTensor(x_train)
        self.y_data = torch.LongTensor(y_train)        
    def __getitem__(self, index):
        # 根据索引返回数据和对应的标签
        return self.x_data[index], self.y_data[index]
    def __len__(self): 
        # 返回文件数据的数目
        return self.len

# 创建 trainloader 和 testloader
trainset = TrainDS()
train_loader = torch.utils.data.DataLoader(dataset=trainset, batch_size=128, shuffle=True, num_workers=2)

**定义网络模型（Context-Gated Convolution （backbone：ResNet50））**

In [9]:
'''
卷积注意力模块
'''
class Conv2d(nn.Conv2d):

    def __init__(self, in_channels, out_channels, kernel_size, stride=1,
                 padding=0, dilation=1, groups=1, bias=True):
        super(Conv2d, self).__init__(in_channels, out_channels, kernel_size, stride,
                 padding, dilation, groups, bias)
        # for convolutional layers with a kernel size of 1, just use traditional convolution
        if kernel_size == 1:
            self.ind = True
        else:
            self.ind = False            
            self.oc = out_channels
            self.ks = kernel_size
            
            # the target spatial size of the pooling layer
            ws = kernel_size
            self.avg_pool = nn.AdaptiveAvgPool2d((ws,ws))
            
            # the dimension of the latent repsentation
            num_lat = int((kernel_size * kernel_size) / 2 + 1)
            
            # the context encoding module
            self.ce = nn.Linear(ws*ws, num_lat, False)            
            self.ce_bn = nn.BatchNorm1d(in_channels)
            self.ci_bn2 = nn.BatchNorm1d(in_channels)
            
            # activation function is relu
            self.act = nn.ReLU(inplace=True)
            
            
            # the number of groups in the channel interacting module
            if in_channels // 16:
                self.g = 16
            else:
                self.g = in_channels
            # the channel interacting module    
            self.ci = nn.Linear(self.g, out_channels // (in_channels // self.g), bias=False)
            self.ci_bn = nn.BatchNorm1d(out_channels)
            
            # the gate decoding module
            self.gd = nn.Linear(num_lat, kernel_size * kernel_size, False)
            self.gd2 = nn.Linear(num_lat, kernel_size * kernel_size, False)
            
            # used to prrepare the input feature map to patches
            self.unfold = nn.Unfold(kernel_size, dilation, padding, stride)
            
            # sigmoid function
            self.sig = nn.Sigmoid()
    def forward(self, x):
        # for convolutional layers with a kernel size of 1, just use traditional convolution
        if self.ind:
            return F.conv2d(x, self.weight, self.bias, self.stride,
                        self.padding, self.dilation, self.groups)
        else:
            b, c, h, w = x.size()
            weight = self.weight
            # allocate glbal information
            gl = self.avg_pool(x).view(b,c,-1)
            # context-encoding module
            out = self.ce(gl)
            # use different bn for the following two branches
            ce2 = out
            out = self.ce_bn(out)
            out = self.act(out)
            # gate decoding branch 1
            out = self.gd(out)
            # channel interacting module
            if self.g >3:
                # grouped linear
                oc = self.ci(self.act(self.ci_bn2(ce2).\
                                      view(b, c//self.g, self.g, -1).transpose(2,3))).transpose(2,3).contiguous()
            else:
                # linear layer for resnet.conv1
                oc = self.ci(self.act(self.ci_bn2(ce2).transpose(2,1))).transpose(2,1).contiguous() 
            oc = oc.view(b,self.oc,-1) 
            oc = self.ci_bn(oc)
            oc = self.act(oc)
            # gate decoding branch 2
            oc = self.gd2(oc)   
            # produce gate
            out = self.sig(out.view(b, 1, c, self.ks, self.ks) + oc.view(b, self.oc, 1, self.ks, self.ks))
            # unfolding input feature map to patches
            x_un = self.unfold(x)
            b, _, l = x_un.size()
            # gating
            out = (out * weight.unsqueeze(0)).view(b, self.oc, -1)
            # currently only handle square input and output
            return torch.matmul(out,x_un).view(b, self.oc, int(np.sqrt(l)), int(np.sqrt(l)))  


def conv3x3(in_planes, out_planes, stride=1):
    """3x3 context gated convolution with padding"""
    return Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                     padding=1, bias=False)

def conv1x1(in_planes, out_planes, stride=1):
    """1x1 convolution"""
    return Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)

class GDNet(nn.Module):
  def __init__(self):
    super(GDNet, self).__init__()
    self.conv_first = conv3x3(3,48)
    self.bn_first = nn.BatchNorm2d(48)

    self.conv_middle = conv3x3(48,96)
    self.bn_middle = nn.BatchNorm2d(96)

    self.conv_final = conv3x3(96,48)
    self.bn_final = nn.BatchNorm2d(48)

    self.linear1 = nn.Linear(3888, 100)
    self.linear2 = nn.Linear(100, 2)

  def forward(self, x):
    ori_out = F.relu(self.bn_first(self.conv_first(x)))

    out = self.conv_middle(ori_out)
    out = F.relu(self.bn_middle(out))

    out = self.conv_final(out)
    out = self.bn_final(out)

    out = out.view(out.size(0), -1)
    # print(out.shape)
    out = self.linear1(out)
    out = self.linear2(out)

    return out

训练数据增强（Mix Up）

In [10]:
def mixup_data(x, y, alpha=0.5):#, use_cuda=True):

    '''Compute the mixup data. Return mixed inputs, pairs of targets, and lambda'''
    if alpha > 0.:
        lam = np.random.beta(alpha, alpha)
        lam = max(lam, 1-lam)
    else:
        lam = 1.
    batch_size = x.size()[0]
    #if use_cuda:
    index = torch.randperm(batch_size)#.to(device)#.cuda()
    #else:
    #    index = torch.randperm(batch_size)
    mixed_x = lam * x + (1 - lam) * x[index,:]
    y_a, y_b = y, y[index]


    return mixed_x, y_a, y_b, lam

def mixup_criterion(y_a, y_b, lam):
    # sigmoid = 1.0/(1 + math.exp( 5 - 10*lam))
    # sigmoid = 4.67840515/(5.85074311 + math.exp(6.9-10.2120858*lam))
    # sigmoid = 1.531 /(1.71822 + math.exp(6.9-12.2836*lam))
    # return lambda criterion, pred: sigmoid * criterion(pred, y_a) + (1 - sigmoid) * criterion(pred, y_b)

    return lambda criterion, pred: lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

def adjust_learning_rate(optimizer, epoch):#epoch=100
    """decrease the learning rate at 100 and 150 epoch"""
    lr = base_learning_rate
    if epoch <= 9 and lr > 0.1:
        # warm-up training for large minibatch
        lr = 0.1 + (base_learning_rate - 0.1) * epoch / 10.
    if epoch >= 50 :
        lr /= 10
    if epoch >= 75 :
        lr /= 10

训练

In [11]:
# 使用GPU训练，可以在菜单 "代码执行工具" -> "更改运行时类型" 里进行设置
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
alpha = 0.5

# 网络放到GPU上
net = GDNet().to(device)
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.SGD(net.parameters(), lr=0.001,momentum=0.9)
base_learning_rate=0.001
best_acc=0

# 开始训练
for epoch in range(15):
    net.train()
    train_loss = 0.0
    correct = 0.0
    total = 0
    #adjust_learning_rate(optimizer, epoch)
    for batch_idx1, (inputs, targets) in enumerate(train_loader):
        #if use_cuda:
        inputs, targets = inputs.to(device), targets.to(device)
        #print(inputs)
        mask = random.random()        
       
        if epoch >= 5 and epoch< 10:
            threshold = (9 - epoch) / (9 - 5)
            if mask < threshold:
                inputs, targets_a, targets_b, lam = mixup_data(inputs, targets, alpha)#, use_cuda)
            else:
                targets_a, targets_b = targets, targets
                lam = 1.0
        elif epoch >= 10:
            targets_a, targets_b = targets, targets
            lam = 1.0
        else:
            inputs, targets_a, targets_b, lam = mixup_data(inputs, targets, alpha)#, use_cuda)

        # 优化器梯度归零
        optimizer.zero_grad()
        #inputs, targets_a, targets_b = Variable(inputs), Variable(targets_a), Variable(targets_b)
        # 正向传播 +　反向传播 + 优化 
        outputs = net(inputs)
        # print(outputs.shape)
        loss_func = mixup_criterion(targets_a, targets_b, lam)
        loss = loss_func(criterion, outputs)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += targets.size(0)
        correct += lam * predicted.eq(targets_a.data).cpu().sum().item() + (1.0 - lam) * predicted.eq(targets_b.data).cpu().sum().item()


    print('Epoch: %d | Loss: %.3f | Acc: %.3f%% (%d/%d)' % (epoch+1, train_loss/(epoch+1), 100.*correct/total, correct, total))

Epoch: 1 | Loss: 13.968 | Acc: 93.879% (9387/10000)
Epoch: 2 | Loss: 4.908 | Acc: 95.597% (9559/10000)
Epoch: 3 | Loss: 2.986 | Acc: 96.024% (9602/10000)
Epoch: 4 | Loss: 2.372 | Acc: 95.287% (9528/10000)
Epoch: 5 | Loss: 1.896 | Acc: 95.398% (9539/10000)
Epoch: 6 | Loss: 1.575 | Acc: 95.486% (9548/10000)
Epoch: 7 | Loss: 0.956 | Acc: 97.023% (9702/10000)
Epoch: 8 | Loss: 0.805 | Acc: 96.877% (9687/10000)
Epoch: 9 | Loss: 0.376 | Acc: 98.882% (9888/10000)
Epoch: 10 | Loss: 0.072 | Acc: 99.890% (9989/10000)
Epoch: 11 | Loss: 0.034 | Acc: 99.890% (9989/10000)
Epoch: 12 | Loss: 0.015 | Acc: 99.990% (9999/10000)
Epoch: 13 | Loss: 0.014 | Acc: 99.980% (9998/10000)
Epoch: 14 | Loss: 0.012 | Acc: 99.960% (9996/10000)
Epoch: 15 | Loss: 0.013 | Acc: 99.940% (9994/10000)


测试

In [12]:
# 逐像素预测类别
net.eval()
outputs = np.zeros( (ylen, xlen) )

for i in range(ylen):
    for j in range(xlen):
        if preclassify_lab[i, j] != 1.5 :
            outputs[i, j] = preclassify_lab[i, j]
        else:
            img_patch = x_test[i*xlen+j, :, :, :]
            img_patch = img_patch.reshape(1, img_patch.shape[0], img_patch.shape[1], img_patch.shape[2])
            img_patch = torch.FloatTensor(img_patch).to(device)
            prediction = net(img_patch)
            prediction = np.argmax(prediction.detach().cpu().numpy(), axis=1)
            outputs[i, j] = prediction+1
    if (i+1) % 50 == 0:
        print('... ... row', i+1, ' handling ... ...')

outputs = outputs-1

# plt.imshow(outputs, 'gray')

res = outputs*255
res = postprocess(res)
evaluate(im_gt, res)


save_dir = os.path.join('result', 'GDNet')
os.makedirs(save_dir, exist_ok=True)
final_png = os.path.join(save_dir, 'Sulzberger.png')
plt.imsave(final_png, res, cmap='gray')
print(f'最终结果图已保存到: {final_png}')


... ... row 50  handling ... ...


/tmp/ipykernel_2122450/3008126559.py:15: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  outputs[i, j] = prediction+1


... ... row 100  handling ... ...
... ... row 150  handling ... ...
... ... row 200  handling ... ...
... ... row 250  handling ... ...
 Change detection results ==>
 ... ... FA:   1554
 ... ... MA:   617
 ... ... OE:   2171
 ... ... PCC:  96.69 %
 ... ... Kappa:  0.9132
最终结果图已保存到: result/GDNet/Sulzberger.png
